In [12]:
conda create -n sales_streamlit python=3.11 pandas numpy scikit-learn=1.3.2 streamlit -y

Jupyter detected...
3 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: win-64
Solving environment: failed

Note: you may need to restart the kernel to use updated packages.



PackagesNotFoundError: The following packages are not available from current channels:

  - scikit-learn=1.3.2

Current channels:

  - defaults

To search for alternate channels that may provide the conda package you're
looking for, navigate to

    https://anaconda.org

and use the search bar at the top of the page.




In [13]:
#Jupyter Notebook Program:

import pickle
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import sklearn
print(sklearn.__version__)

1.7.2


In [14]:
# -----------------------------
# 1. Load data
# -----------------------------
data = pd.read_csv("cust_latest.csv")


In [15]:
# -----------------------------
# 2. Clean and prepare date
# -----------------------------
data["PO Date"] = pd.to_datetime(data["PO Date"])

data["day"] = data["PO Date"].dt.day
data["month"] = data["PO Date"].dt.month
data["year"] = data["PO Date"].dt.year


In [16]:
# -----------------------------
# 3. Define X and y
# -----------------------------
target_col = "Bill Qty"

feature_cols = [
"day",
"month",
"year",
"Brand Name",
"Material Family"
]

X = data[feature_cols]
y = data[target_col]

In [17]:
# -----------------------------
# 4. Identify column types
# -----------------------------
numeric_features = ["day", "month", "year"]
categorical_features = ["Brand Name", "Material Family"]

preprocessor = ColumnTransformer(
transformers=[
("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
("num", "passthrough", numeric_features)
]
)


In [18]:
# -----------------------------
# 5. Train/test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.33, random_state=42
)


In [19]:
# -----------------------------
# 6. Create models
# -----------------------------
lin_model = Pipeline([
("preprocessor", preprocessor),
("model", LinearRegression())
])

rf_model = Pipeline([
("preprocessor", preprocessor),
("model", RandomForestRegressor(
n_estimators=100,
max_depth=8,
random_state=42
))
])

svr_model = Pipeline([
("preprocessor", preprocessor),
("model", SVR(C=10, epsilon=0.1, kernel="rbf"))
])

voting_model = VotingRegressor([
("lin", lin_model),
("rf", rf_model),
("svr", svr_model)
])

models = {
"LinearRegression": lin_model,
"RandomForest": rf_model,
"SVR": svr_model,
"VotingRegressor": voting_model
}


In [20]:
# -----------------------------
# 7. Train and evaluate
# -----------------------------
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    print(f"{name} MSE: {mse:.4f}")

LinearRegression MSE: 563.9663
RandomForest MSE: 564.8123
SVR MSE: 659.8798
VotingRegressor MSE: 516.6718


In [21]:
# -----------------------------
# 8. Save model
# -----------------------------
with open("predicted_sales_model.pkl", "wb") as f:
    pickle.dump({
    "model": voting_model,
    "features": feature_cols,
    "brand_options": sorted(data["Brand Name"].dropna().unique()),
    "material_options": sorted(data["Material Family"].dropna().unique())
}, f)

print("Saved predicted_sales_model.pkl")

Saved predicted_sales_model.pkl
